### Facteurs de perturbation des forêts de tourbières du Congo, depuis 2022
*Conception de l'analyse d'après Nesha et al. (2024), Environ. Res. Lett. 19 104031, [doi:10.1088/1748-9326/ad6679](https://doi.org/10.1088/1748-9326/ad6679), à partir des alertes du WUR attribuées à un facteur de perturbation.*

| Étape | Contenu |
|---|---|
| Perturbations | Cartes des facteurs du WUR uniquement (alertes intégrées RADD + GLAD-L + GLAD-S2, + DIST à partir de 2026) : historique 2022–2025 + opérationnel 2026 → |
| Zone d'étude | Occupation du sol : **tourbière dominée par les palmiers (4)**, **tourbière dominée par les feuillus (5)**, **zone tampon (6)**, limitée au pays choisi |
| Contexte | Aires protégées, concessions forestières, unités administratives (par polygone) |
| Analyses | Facteurs et superficies par classe · aires protégées, concessions, unités administratives · taille des événements · profils mensuels · points chauds à 1 km · distance au bord de la tourbière, aux routes et aux rivières |

**Mode d'emploi (utilisateurs).** Indiquez dans `ASSET_ROOT` (§1) un dossier de votre propre projet Earth Engine, puis exécutez tout (Run All). Le notebook lit la pile partagée (`STACK_ASSET`), exporte les statistiques, les événements et la grille de 1 km dans votre dossier (8b), attend qu'ils soient prêts, puis produit tous les tableaux, graphiques et cartes (enregistrés dans `outputs_peat_disturbance/`). Vous devez avoir accès en lecture à la pile partagée et aux couches d'entrée listées au §3.

**Mise à jour trimestrielle (propriétaire de la pile).** Mettez `OVERWRITE = True` et exécutez tout : 8a reconstruit la pile dans `STACK_ASSET` (≈1–2 h ; elle est indisponible pour les utilisateurs pendant ce temps), puis 8b reconstruit les sorties. Remettez ensuite `False`. Une pile qui n'existe pas encore est toujours construite. Quiconque exécute le notebook ensuite obtient automatiquement des sorties à jour : les sorties plus anciennes que la pile sont réexportées, et les tableaux téléchargés plus anciens que leur asset sont téléchargés à nouveau.

#### 0 · Initialisation

In [ ]:
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display
import ee

EE_PROJECT = "fao-sepal-361210"   # projet de calcul (SEPAL)
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("API Earth Engine", ee.__version__, "| projet :", EE_PROJECT)

#### 1 · Configuration

In [ ]:
COUNTRY_LIST = ["Democratic Republic of the Congo"]     # ADM0_NAME de FAO GAUL ; ajouter "Congo" pour la République du Congo

In [ ]:
# VOTRE dossier pour les statistiques, les événements et la grille (accès en écriture nécessaire)
ASSET_ROOT = "projects/congopeatlands/assets"

In [ ]:
# ── Emplacement des données ─────────────────────────────────────────────────
STACK_ASSET = "projects/congopeatlands/assets/stack_wur_dem_2022_2026"   # pile partagée (utilisée si elle existe, construite ici sinon)
LOCAL_OUT = Path("outputs_peat_disturbance"); LOCAL_OUT.mkdir(exist_ok=True)
OVERWRITE = False                                       # True = tout reconstruire : la pile (si vous pouvez y écrire) et vos sorties
REFRESH_TABLES = False                                  # True = télécharger à nouveau les tableaux même si la copie locale est à jour

# ── Période et pays (doivent correspondre à la construction de la pile) ─────
START_YEAR = 2022                                       # les facteurs du WUR commencent en janvier 2022
END_YEAR = pd.Timestamp.now().year                      # année en cours (partielle)

# ── Zone d'étude : classes d'occupation du sol ─────────────────────────────
LANDCOVER_ASSET = "projects/congopeatlands/assets/LandCover_Crezee_2022_500mbuffer"
LANDCOVER_CLASSES = {4: "tourbière dominée par les palmiers", 5: "tourbière dominée par les feuillus", 6: "zone tampon"}
PEAT_CLASS_CODES = [4, 5]                               # tourbe (utilisée pour l'analyse de la distance au bord)
ZONE_BUFFER = LANDCOVER_CLASSES.get(6)                  # libellé de la zone tampon (exclue de l'analyse de la distance au bord)
ZONE_COLORS = {"tourbière dominée par les palmiers": "#66c2a4", "tourbière dominée par les feuillus": "#006d2c",
               "zone tampon": "#fdd49e"}

# ── Facteurs de perturbation (WUR) ──────────────────────────────────────────
DRIVERS_HIST_ASSET = "projects/wurnrt-drivers/assets/distribution/driverclassification_afr_202201_202512"
DRIVERS_OPER_ASSET = "projects/wurnrt-drivers/assets/operational_postprocessed_afr_collection"
DRIVER_BAND, DRIVER_DATE_BAND = "class", "date"         # date = AAjjj (année sur 2 chiffres + jour de l'année)
DRIVER_CLASSES = {
    1: "Agriculture à petite échelle", 2: "Agriculture à petite échelle (feu)",
    3: "Agriculture à grande échelle", 4: "Agriculture à grande échelle (feu)",
    5: "Construction de routes", 6: "Exploitation forestière sélective", 7: "Exploitation minière", 8: "Inondation",
    9: "Feu de forêt", 10: "Autre perturbation naturelle", 11: "Classification incertaine"}
FIRE_SUBTYPES = {2: 1, 4: 3}                            # sous-type « avec feu » → classe parente (regroupés dans les graphiques)
DRIVER_GROUP_COLORS = {
    "Agriculture à petite échelle": "#e6ab02", "Agriculture à grande échelle": "#a6761d", "Construction de routes": "#525252",
    "Exploitation forestière sélective": "#1b9e77", "Exploitation minière": "#7570b3", "Inondation": "#1f78b4",
    "Feu de forêt": "#d95f02", "Autre perturbation naturelle": "#66a61e", "Classification incertaine": "#bdbdbd"}

# ── Distances ───────────────────────────────────────────────────────────────
PEAT_DIST_ASSET = "projects/congopeatlands/assets/Crezee_peat_distance"
PEAT_EDGE_BAND = "dist_to_edge_m"                       # dans la tourbe : distance au bord de la tourbière (m)
ROADS_ASSET = "projects/cafi_fao_congo/regional/roads_Kleinschroth_etal_2019"
RIVERS_LINES = "projects/congopeatlands/assets/COD_waterways_lines"        # HOTOSM
WATER_POLY = "projects/congopeatlands/assets/COD_waterways_poly"
RIVER_TYPE_FIELD, RIVER_TYPES = "waterway", None        # ex. ["river", "canal"] ; None = toutes les lignes
ACCESS_SCALE, ACCESS_NEIGH_PX = 100, 300                # distance aux routes/rivières à 100 m, rayon de recherche 30 km
EDGE_BINS_KM = [0.5, 1, 2, 3, 4, 5, 10]
ACCESS_BINS_KM = [1, 2, 3, 4, 5, 10, 15, 20, 25]

# ── Aires protégées, concessions, unités administratives ───────────────────
PA_ASSET = "projects/congopeatlands/assets/peatlands_PAs"
CONCESSIONS_ASSET = "projects/congopeatlands/assets/forest_concessions_COD"
ADMIN_ASSET = "projects/congopeatlands/assets/peatlands_adm1"
PA_NAME_FIELD = "NAME"                     # propriété contenant le nom de l'aire protégée (None = détection automatique)
CONC_NAME_FIELD = "attributai"             # propriété contenant le nom de la concession (attributaire)
ADMIN_NAME_FIELD = "gaul1_name"            # propriété du nom des unités administratives (ex. "gaul1_name" ; None = détection automatique)

# ── Paramètres d'analyse (comme dans Nesha et al. 2024) ────────────────────
SIZE_CLASSES_HA = [0.5, 2]                              # petit < 0,5 ≤ moyen < 2 ≤ grand
INTENSITY_THRESHOLDS_PCT = [2, 20]                      # cellule de 1 km : faible < 2 % ≤ moyenne ≤ 20 % < élevée (point chaud)
TEMPORAL_HOTSPOT_PCT = 10                               # mois avec > 10 % des perturbations de l'année
MMU_HA = 0.1                                            # les événements plus petits sont signalés (fragments d'intersection)

# ── Calcul et cartes ────────────────────────────────────────────────────────
TILE_DEG = 0.5                                          # tuiles de calcul (seules les tuiles avec zone d'étude sont utilisées)
TILE_SCALE = 8
WAIT_POLL_MIN, WAIT_MAX_H = 5, 12                       # 8b attend les exports en cours : vérification toutes les 5 min, abandon après 12 h
TEST_BOX = [18.4, 2.8, 18.9, 3.3]                       # zone de test — à déplacer si elle ne contient pas d'alertes
MAP_RES_DEG = 0.01                                      # résolution la plus fine des cartes statiques (~1 km)
MAP_BASEMAP = "Esri.WorldTopoMap"                       # aperçu interactif : "Esri.WorldTopoMap" | "Esri.WorldImagery"
DOM_CELL_DEG = 0.1                                      # taille de cellule de la carte du facteur dominant
GRID_DEG = 0.009                                        # cellule de la grille des points chauds ≈ 1 km à l'équateur

# ── Exports locaux pour les SIG et les tableaux de bord (§17) ───────────────
EXPORT_GEOTIFF = True                                   # GeoTIFF multibande (~1 km) des cartes de la grille
EXPORT_DASHBOARD_TABLES = True                          # CSV structurés avec coordonnées / clés de jointure

# ── Valeurs dérivées ────────────────────────────────────────────────────────
ASSET_ROOT = ASSET_ROOT.rstrip("/")
START_YEAR = max(START_YEAR, 2022)
YEARS = list(range(START_YEAR, END_YEAR + 1)); N_YEARS = len(YEARS)
ZONE_CODES = list(LANDCOVER_CLASSES)                    # indice de zone = position + 1
ZONE_ORDER = list(LANDCOVER_CLASSES.values())
RUN_TAG = STACK_ASSET.rstrip("/").split("/")[-1].replace("stack_", "", 1)   # les noms des sorties suivent celui de la pile
print(f"Pile : {STACK_ASSET}\nSorties : {ASSET_ROOT}/{{stats,events,grid1km_tbl}}_{RUN_TAG}")

#### 2 · Fonctions utilitaires

In [ ]:
def asset_type(path):
    """'IMAGE', 'IMAGE_COLLECTION', 'TABLE', 'FOLDER' ou None (absent / pas d'accès)."""
    if not path:
        return None
    path = path.rstrip("/")
    try:
        if path.endswith("/assets"):
            ee.data.listAssets({"parent": path, "pageSize": 1}); return "FOLDER"
        return ee.data.getAsset(path)["type"]
    except (ee.EEException, TypeError):
        return None

def asset_time(path):
    """Date de dernière mise à jour d'un asset (UTC) ou None."""
    try:
        return pd.Timestamp(ee.data.getAsset(path.rstrip("/"))["updateTime"])
    except Exception:
        return None

def geom_rings(g):
    """Tous les anneaux extérieurs / lignes d'une géométrie GeoJSON sous forme de tableaux (Polygon, MultiPolygon, GeometryCollection, lignes)."""
    t, c = (g or {}).get("type"), (g or {}).get("coordinates")
    if t == "Polygon": return [np.array(c[0])]
    if t == "MultiPolygon": return [np.array(p[0]) for p in c]
    if t == "LineString": return [np.array(c)]
    if t == "MultiLineString": return [np.array(l) for l in c]
    if t == "GeometryCollection": return [r for sub in g.get("geometries", []) for r in geom_rings(sub)]
    return []

def ensure_folder(path):
    if path.rstrip("/").endswith("/assets"):
        return
    if asset_type(path) is None:
        ee.data.createAsset({"type": "FOLDER"}, path); print("dossier créé", path)

def load_image(asset, band=None):
    t = asset_type(asset)
    if t not in ("IMAGE", "IMAGE_COLLECTION"):
        return None
    img = ee.ImageCollection(asset).mosaic() if t == "IMAGE_COLLECTION" else ee.Image(asset)
    return img.select(band) if band else img.select(0)

def yydoy_to_ts(v):
    v = int(v)
    return pd.Timestamp(f"{2000 + v // 1000}-01-01") + pd.Timedelta(days=v % 1000 - 1)

def doy_to_month(year_img, doy_img):
    leap = year_img.mod(4).eq(0)
    doy = doy_img.subtract(leap.And(doy_img.gt(59)))
    month = ee.Image(1)
    for cum in [31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]:
        month = month.add(doy.gt(cum))
    return month.int()

def bin_index(img, edges):
    b = ee.Image(0)
    for e in edges:
        b = b.add(img.gt(e))
    return b.int()

def cum_labels(edges):
    return [f"≤{e:g} km" for e in edges] + ["tout"]

def distance_km(source, scale, neigh_px):
    """Distance euclidienne (km) au pixel non nul le plus proche de `source`, à `scale` m ; au-delà du rayon = « loin »."""
    far = neigh_px * scale / 1000.0
    d = (source.unmask(0).gt(0)
         .fastDistanceTransform(neigh_px, "pixels", "squared_euclidean")
         .sqrt().multiply(ee.Image.pixelArea().sqrt()).divide(1000)
         .reproject(crs="EPSG:4326", scale=scale))
    return d.unmask(far)

def encode(imgs, fields):
    code = ee.Image(0).int()
    for name, size in fields:
        code = code.multiply(size).add(ee.Image(imgs[name]).int())
    return code.int()

def decode(codes, fields):
    rem = pd.Series(codes).round().astype("int64").to_numpy().copy()
    out = {}
    for name, size in reversed(fields):
        out[name] = rem % size
        rem //= size
    return pd.DataFrame(out)[[n for n, _ in fields]]

TASKS = {}
def active_tasks():
    """{description: (état, id)} des tâches Earth Engine READY / RUNNING de ce compte (y compris de sessions précédentes)."""
    out = {}
    try:
        for op in ee.data.listOperations():
            md = op.get("metadata", {})
            if md.get("state") in ("PENDING", "RUNNING", "CANCELLING"):
                out[md.get("description")] = (md.get("state"), op.get("name", "").split("/")[-1])
    except Exception as e:
        print("⚠ impossible de lister les tâches en cours :", e)
    return out

def start_export(name, make_task, asset_id, overwrite=None, newer_than=None):
    """Lance un export sauf si l'asset existe et est à jour (ou si la même tâche est déjà en cours)."""
    overwrite = OVERWRITE if overwrite is None else overwrite
    short = asset_id.split("/assets/")[-1]
    if asset_type(asset_id) is not None:
        stale = newer_than is not None and (asset_time(newer_than) or pd.Timestamp.min.tz_localize("UTC")) > asset_time(asset_id)
        if not (overwrite or stale):
            print(f"[ignoré] {short} existe et est à jour"); return None
        print(f"[reconstruction] {short} — {'la pile est plus récente' if stale else 'écrasement demandé'}")
        ee.data.deleteAsset(asset_id)
    task = make_task()
    desc = task.config.get("description") if hasattr(task, "config") else None
    running = active_tasks()
    if desc in running:
        print(f"[ignoré] {desc} est déjà {running[desc][0]} ({running[desc][1]}) — pas de doublon lancé"); return None
    task.start(); TASKS[name] = task
    print(f"[lancé] {name} : {task.id}"); return task

def fc_to_df(fc):
    return ee.data.computeFeatures({"expression": ee.FeatureCollection(fc), "fileFormat": "PANDAS_DATAFRAME"})

def table_df(asset, selectors, refresh=None):
    """Table EE enregistrée → DataFrame via un seul téléchargement CSV, mis en cache dans LOCAL_OUT/ee_tables (téléchargé à nouveau si l'asset est plus récent)."""
    import urllib.request
    cache = LOCAL_OUT / "ee_tables"; cache.mkdir(exist_ok=True)       # téléchargements bruts, séparés des CSV de résultats
    path = cache / f"{asset.split('/')[-1]}.csv"
    if refresh is None:
        refresh = REFRESH_TABLES
    if path.exists() and not refresh:                 # asset plus récent que la copie locale → nouveau téléchargement
        t = asset_time(asset)
        refresh = t is not None and t > pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC")
    if refresh or not path.exists():
        n = ee.FeatureCollection(asset).size().getInfo()
        print(f"téléchargement de {asset.split('/')[-1]} ({n:,} lignes) …", end=" ", flush=True)
        url = ee.FeatureCollection(asset).getDownloadURL(filetype="csv", selectors=selectors,
                                                         filename=path.stem)
        urllib.request.urlretrieve(url, path)
        print("terminé")
    df = pd.read_csv(path, low_memory=False)
    for c in df.columns:                              # nombres stockés en texte (ex. valeurs vides / nulles) → numériques
        if c in (".geo", "kind") or pd.api.types.is_numeric_dtype(df[c]):
            continue
        num = pd.to_numeric(df[c], errors="coerce")
        bad = num.isna() & df[c].notna()
        if bad.any():
            print(f"⚠ {path.stem}.{c} : {bad.sum():,} valeurs non numériques mises à manquant, ex. {df.loc[bad, c].unique()[:5].tolist()}")
        df[c] = num
    if ".geo" in df:
        xy = df.pop(".geo").map(lambda s: json.loads(s)["coordinates"])
        df["lon"], df["lat"] = xy.str[0], xy.str[1]
    return df

def pct(s):
    return s / s.sum() * 100

def cum_share(df, col, edges, weight=None):
    """% cumulé par classe de distance ; weight=None compte les lignes (événements), sinon additionne cette colonne."""
    n = len(edges) + 1
    s = (df.groupby(col)[weight].sum() if weight else df.groupby(col).size()).reindex(range(n), fill_value=0)
    return pd.Series((s.cumsum() / max(s.sum(), 1e-9) * 100).to_numpy(), index=cum_labels(edges))

NAME_CANDIDATES = ["NAME", "Name", "name", "NOM", "Nom", "nom", "ORIG_NAME", "gaul1_name", "GAUL1_NAME", "ADM1_NAME",
                   "ADM1_FR", "ADM1_EN", "NAME_1", "shapeName", "DESIG", "attributai", "concession", "Concession",
                   "titulaire", "TITULAIRE"]
def name_field(fc, preferred=None):
    props = [p for p in ee.Feature(ee.FeatureCollection(fc).first()).propertyNames().getInfo() if not p.startswith("system:")]
    if preferred:
        if preferred not in props:
            raise ValueError(f"{preferred!r} introuvable ; propriétés disponibles : {props}")
        return preferred
    return next((c for c in NAME_CANDIDATES if c in props), props[0] if props else None)

from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
def plain_log(axis):
    """Étiquette un axe logarithmique avec des nombres simples (0.1, 1, 10, 1,000) au lieu de puissances de dix."""
    axis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}" if v >= 1 else f"{v:g}"))
    axis.set_minor_formatter(NullFormatter())

FIG_DIR = LOCAL_OUT / "figures"; FIG_DIR.mkdir(exist_ok=True)
def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}_{RUN_TAG}.png", dpi=200, bbox_inches="tight")

#### 3 · Vérification des assets et des cartes des facteurs

In [ ]:
ASSETS = {"Pile partagée": STACK_ASSET, "Occupation du sol (obligatoire)": LANDCOVER_ASSET, "Distance à la tourbe": PEAT_DIST_ASSET,
          "Facteurs, historique": DRIVERS_HIST_ASSET, "Facteurs, opérationnel": DRIVERS_OPER_ASSET,
          "Routes": ROADS_ASSET, "Cours d'eau (lignes)": RIVERS_LINES, "Surfaces en eau (polygones)": WATER_POLY,
          "Aires protégées": PA_ASSET, "Concessions forestières": CONCESSIONS_ASSET, "Unités administratives": ADMIN_ASSET,
          "Dossier de sortie": ASSET_ROOT}
chk = pd.DataFrame([{"couche": k, "asset": a.split("/assets/")[-1], "type": asset_type(a) or "✗ illisible"}
                    for k, a in ASSETS.items()]).set_index("couche")
display(chk)
if chk.loc["Occupation du sol (obligatoire)", "type"].startswith("✗"):
    raise RuntimeError("La couche d'occupation du sol est obligatoire — corrigez le chemin au §1 ou demandez l'accès en lecture")
STACK_EXISTS = chk.loc["Pile partagée", "type"] == "IMAGE"
BUILD_STACK = OVERWRITE or not STACK_EXISTS             # reconstruite sur demande, construite si absente (8a)
if not STACK_EXISTS:
    print(f"⚠ Aucune pile lisible dans {STACK_ASSET} — elle sera CONSTRUITE à cet emplacement en 8a (≈1–2 h, accès en écriture nécessaire).\n"
          f"  Si vous vouliez utiliser une pile existante, vérifiez le chemin ou demandez l'accès en lecture à son propriétaire.")
elif OVERWRITE:
    print(f"↻ OVERWRITE = True — la pile {STACK_ASSET} sera RECONSTRUITE en 8a (si vous avez l'accès en écriture)")
else:
    print(f"✓ Utilisation de la pile existante {STACK_ASSET}")
if BUILD_STACK and chk.loc["Distance à la tourbe", "type"].startswith("✗"):
    raise RuntimeError("La couche de distance à la tourbe est nécessaire pour construire la pile")

# collections des facteurs → une seule collection avec 'class' et 'date'
dcol, rows = None, []
for name, asset in [("historique", DRIVERS_HIST_ASSET), ("opérationnel", DRIVERS_OPER_ASSET)]:
    t = asset_type(asset)
    col = (ee.ImageCollection(asset) if t == "IMAGE_COLLECTION" else
           ee.ImageCollection([ee.Image(asset)]) if t == "IMAGE" else None)
    row = {"collection": name, "type": t or "✗ illisible"}
    if col is not None:
        info = ee.Dictionary({"n": col.size(), "bands": col.first().bandNames(),
                              "t0": col.aggregate_min("system:time_start"),
                              "t1": col.aggregate_max("system:time_start")}).getInfo()
        to_d = lambda ms: pd.to_datetime(ms, unit="ms").date() if ms is not None else "–"
        row.update({"images": info["n"], "bandes": ", ".join(info["bands"]),
                    "première image": to_d(info["t0"]), "dernière image": to_d(info["t1"])})
        col = col.map(lambda i: i.select([DRIVER_BAND, DRIVER_DATE_BAND]).toInt32())
        dcol = col if dcol is None else dcol.merge(col)
    rows.append(row)
display(pd.DataFrame(rows).set_index("collection").fillna("–"))
if dcol is None and BUILD_STACK:
    raise RuntimeError("Aucune des deux collections de facteurs n'est lisible — nécessaire pour construire la pile")

### Partie A · Construction des couches dans Earth Engine

#### 4 · Zone d'étude : classes d'occupation du sol, pays, tuiles de calcul
Chaque pixel reçoit un indice de zone : 1 = tourbière dominée par les palmiers (classe 4), 2 = tourbière dominée par les feuillus (5), 3 = zone tampon (6). Tout est masqué au pays choisi, et seules les tuiles contenant de la zone d'étude sont traitées.

In [ ]:
gaul0 = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.inList("ADM0_NAME", COUNTRY_LIST))
found = gaul0.aggregate_array("ADM0_NAME").distinct().getInfo()
assert set(found) == set(COUNTRY_LIST), f"Absent de GAUL : {set(COUNTRY_LIST) - set(found)}"
in_country = ee.Image(0).byte().paint(gaul0, 1)

lc = load_image(LANDCOVER_ASSET)
LC_PROJ = lc.projection()
zone = (lc.remap(ZONE_CODES, list(range(1, len(ZONE_CODES) + 1)), 0).unmask(0)
        .multiply(in_country).int().rename("zone"))                              # 0 = hors zone d'étude
peat_mask = lc.remap(PEAT_CLASS_CODES, [1] * len(PEAT_CLASS_CODES), 0).unmask(0).And(in_country)
study = zone.gt(0)

# tuiles sur l'emprise de l'occupation du sol ∩ pays, en ne gardant que celles avec zone d'étude
box = lc.geometry().bounds().intersection(gaul0.geometry().bounds(), 100).bounds()
ring = box.coordinates().getInfo()[0]
x0, x1 = math.floor(min(p[0] for p in ring)), math.ceil(max(p[0] for p in ring))
y0, y1 = math.floor(min(p[1] for p in ring)), math.ceil(max(p[1] for p in ring))
grid_tiles = [ee.Feature(ee.Geometry.Rectangle([x, y, x + TILE_DEG, y + TILE_DEG], None, False), {"tid": i})
              for i, (x, y) in enumerate((x, y) for x in np.arange(x0, x1, TILE_DEG) for y in np.arange(y0, y1, TILE_DEG))]
tiles_all = ee.FeatureCollection(grid_tiles)
counted = study.selfMask().reduceRegions(collection=tiles_all, reducer=ee.Reducer.count(), scale=500, tileScale=4)
tiles_fc = counted.filter(ee.Filter.gt("count", 0)).select(["tid"])
rects = [f["geometry"]["coordinates"][0] for f in tiles_fc.getInfo()["features"]]
xmin = min(p[0] for r in rects for p in r); xmax = max(p[0] for r in rects for p in r)
ymin = min(p[1] for r in rects for p in r); ymax = max(p[1] for r in rects for p in r)
EXTENT_BOX = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax], None, False)
STUDY_REGION = tiles_fc.geometry()

print(pd.DataFrame({"valeur": [", ".join(COUNTRY_LIST), len(grid_tiles), len(rects), f"{TILE_DEG}°",
                              f"{xmin:g}, {ymin:g}, {xmax:g}, {ymax:g}"]},
                   index=["pays", "tuiles sur l'occupation du sol", "tuiles avec zone d'étude (traitées)", "taille des tuiles",
                          "emprise (O, S, E, N)"]).to_string())

#### 5 · Couche des perturbations : alertes du WUR attribuées à un facteur
Les cartes mensuelles des facteurs sont combinées pixel par pixel, en gardant l'alerte la plus récente et sa classe. Un pixel est une perturbation si la date de son alerte est comprise entre `START_YEAR` et `END_YEAR` et s'il se trouve dans la zone d'étude.

In [ ]:
if BUILD_STACK:
    drv = dcol.map(lambda i: i.updateMask(i.select(DRIVER_BAND).gt(0))).qualityMosaic(DRIVER_DATE_BAND)
    drv_class, drv_date = drv.select(DRIVER_BAND).int(), drv.select(DRIVER_DATE_BAND).int()
    alert_year = drv_date.divide(1000).floor().add(2000).int()
    dist_mask = (drv_class.gt(0).And(alert_year.gte(START_YEAR)).And(alert_year.lte(END_YEAR))
                 .And(study)).selfMask().rename("dist")
    latest = drv_date.reduceRegion(ee.Reducer.max(), EXTENT_BOX, scale=500, maxPixels=1e10,
                                   bestEffort=True).getInfo().get(DRIVER_DATE_BAND)
else:   # dernière alerte enregistrée dans la pile partagée
    latest = ee.Image(STACK_ASSET).select("t").reduceRegion(ee.Reducer.max(), EXTENT_BOX, scale=500, maxPixels=1e10,
                                                           bestEffort=True).getInfo().get("t")
LATEST_ALERT_DATE = yydoy_to_ts(latest).strftime("%Y-%m-%d") if latest else f"{END_YEAR}-12-31"

CLASS_CODES = sorted(DRIVER_CLASSES)
DRIVER_LABELS = ["code non répertorié"] + [DRIVER_CLASSES[c] for c in CLASS_CODES]
ND = len(DRIVER_LABELS)
PARENT = {DRIVER_CLASSES[k]: DRIVER_CLASSES[v] for k, v in FIRE_SUBTYPES.items()}
if BUILD_STACK:
    drv_idx = drv_class.remap(CLASS_CODES, list(range(1, ND)), 0).rename("driver")

tbl = pd.DataFrame({"code WUR": CLASS_CODES, "libellé": [DRIVER_CLASSES[c] for c in CLASS_CODES],
                    "groupe (graphiques)": [PARENT.get(DRIVER_CLASSES[c], DRIVER_CLASSES[c]) for c in CLASS_CODES]})
def _swatch(row):
    col = DRIVER_GROUP_COLORS.get(row["groupe (graphiques)"])
    return [f"background-color: {col}" if c == "groupe (graphiques)" and col else "" for c in row.index]
print(f"Dernière alerte WUR dans l'emprise d'étude : {LATEST_ALERT_DATE}" + ("" if BUILD_STACK else " (d'après la pile partagée)"))
try:
    display(tbl.set_index("code WUR").style.apply(_swatch, axis=1))
except Exception:
    display(tbl.set_index("code WUR"))

#### 6 · Distances au bord de la tourbière, aux routes et aux rivières
* **Bord de la tourbière :** `dist_to_edge_m` précalculé (distance, à l'intérieur de la tourbe, jusqu'à son bord ; 0 dans la zone tampon).
* **Routes :** Kleinschroth et al. (2019). **Rivières :** lignes de cours d'eau HOTOSM + polygones de surfaces en eau. Les deux sont rastérisés, masqués au pays, puis convertis en distance à 100 m.

In [ ]:
edge_km = ee.Image(PEAT_DIST_ASSET).select(PEAT_EDGE_BAND).divide(1000).unmask(0).rename("edge_km")

def painted(asset, width=None, flt=None):
    if asset_type(asset) != "TABLE":
        return None
    fc = ee.FeatureCollection(asset).filterBounds(EXTENT_BOX)
    if flt is not None:
        fc = fc.filter(flt)
    return ee.Image(0).byte().paint(fc, 1, width) if width else ee.Image(0).byte().paint(fc, 1)

road_src = painted(ROADS_ASSET, 1)
river_l = painted(RIVERS_LINES, 1, ee.Filter.inList(RIVER_TYPE_FIELD, RIVER_TYPES) if RIVER_TYPES else None)
water_p = painted(WATER_POLY)
river_src = None
for r in (river_l, water_p):
    if r is not None:
        river_src = r if river_src is None else river_src.Or(r)

FAR_KM = ACCESS_NEIGH_PX * ACCESS_SCALE / 1000
road_km = (distance_km(road_src.multiply(in_country), ACCESS_SCALE, ACCESS_NEIGH_PX) if road_src is not None
           else ee.Image(FAR_KM)).rename("road_km")
river_km = (distance_km(river_src.multiply(in_country), ACCESS_SCALE, ACCESS_NEIGH_PX) if river_src is not None
            else ee.Image(FAR_KM)).rename("river_km")

display(pd.DataFrame({"état": ["✓" if x is not None else "✗ absent → « loin » partout"
                                 for x in (road_src, river_l, water_p)]},
                     index=["routes", "cours d'eau (lignes)", "surfaces en eau (polygones)"]))

#### 6b · Aires protégées, concessions forestières, unités administratives
Chaque couche est rastérisée avec un numéro de polygone (0 = aucun), masquée au pays, afin de pouvoir présenter les résultats par aire protégée, par concession et par unité administrative. Les polygones portant le même nom sont regroupés (pour les concessions, le nom est celui de l'attributaire : `attributai`).

In [ ]:
def id_layer(asset, preferred):
    """Rastérise les polygones avec un identifiant (1..n, par nom distinct) ; renvoie l'image, les noms et le champ."""
    if asset_type(asset) != "TABLE":
        return ee.Image(0).int(), [], None
    fc = ee.FeatureCollection(asset).filterBounds(EXTENT_BOX)
    f = name_field(fc, preferred)
    fc = fc.filter(ee.Filter.notNull([f]))
    names = [n for n in fc.aggregate_array(f).distinct().getInfo() if n not in (None, "")]
    if not names:
        return ee.Image(0).int(), [], f
    fc = fc.remap(names, list(range(1, len(names) + 1)), f)
    return ee.Image(0).int().paint(fc, f).multiply(in_country).int(), [str(n) for n in names], f

PA_IMG, PA_NAMES, PA_FIELD = id_layer(PA_ASSET, PA_NAME_FIELD)
CONC_IMG, CONC_NAMES, CONC_FIELD = id_layer(CONCESSIONS_ASSET, CONC_NAME_FIELD)
ADM_IMG, ADM_NAMES, ADM_FIELD = id_layer(ADMIN_ASSET, ADMIN_NAME_FIELD)
PA_IMG, CONC_IMG, ADM_IMG = PA_IMG.rename("pa"), CONC_IMG.rename("conc"), ADM_IMG.rename("adm")
display(pd.DataFrame({"champ du nom": [PA_FIELD, CONC_FIELD, ADM_FIELD],
                      "polygones (noms distincts) dans l'emprise": [len(PA_NAMES), len(CONC_NAMES), len(ADM_NAMES)]},
                     index=["aires protégées", "concessions forestières", "unités administratives"]).fillna("✗ illisible"))

#### 7 · Test sur une petite zone (quelques secondes)
Vérifie que tout s'évalue correctement avant les exports.

In [ ]:
tb = ee.Geometry.Rectangle(TEST_BOX)
src_img = (ee.Image.pixelArea().divide(1e4).addBands(zone).addBands(drv_idx).updateMask(dist_mask) if BUILD_STACK else
           ee.Image.pixelArea().divide(1e4).addBands(ee.Image(STACK_ASSET).select("zone"))
           .addBands(ee.Image(STACK_ASSET).select("driver")).updateMask(ee.Image(STACK_ASSET).select("driver").mask()))
r = (src_img
     .reduceRegion(ee.Reducer.sum().group(1, "zone").group(2, "driver"), tb, scale=30, maxPixels=1e9).getInfo())
# bandes : 0 superficie, 1 zone, 2 facteur — le groupe externe prend la bande 2 (facteur), le groupe interne regroupe le reste par la bande 1 (zone)
rows = [{"facteur": DRIVER_LABELS[d["driver"]], "zone": ZONE_ORDER[z["zone"] - 1], "ha": z["sum"]}
        for d in r.get("groups", []) for z in d["groups"]]
if rows:
    display(pd.DataFrame(rows).pivot_table(index="facteur", columns="zone", values="ha", aggfunc="sum").round(1))
else:
    print("Aucune alerte dans TEST_BOX — déplacez-la vers un point chaud connu")

#### 8 · Exports (deux étapes)
**8a · Étape 1 — pile (uniquement avec `OVERWRITE = True`, ou si `STACK_ASSET` n'existe pas)** : construit la pile partagée à 10 m des pixels perturbés : facteur, date de l'alerte, classe, aire protégée, concession, unité administrative et les trois distances (mètres). C'est le seul calcul lourd (≈1–2 h). Les utilisateurs sautent cette étape et lisent `STACK_ASSET`.
**8b · Étape 2 — sorties (tout le monde)** : écrit dans `ASSET_ROOT`, réexporte tout ce qui manque ou est plus ancien que la pile, et attend qu'ils soient prêts :
* **stats** : superficie perturbée par année × mois × facteur × classe × classes de distance ; par année × facteur × classe × aire protégée × concession × unité administrative ; et la zone d'étude de chaque combinaison (dénominateurs) ;
* **events** : pixels perturbés connexes (8-connexité) → un point par événement avec superficie, première date et facteur, classe, aire protégée, concession, unité administrative et classes de distance majoritaires ;
* **grid** : un tableau de cellules d'environ 1 km (`GRID_DEG`) avec la zone d'étude (ha) et la superficie perturbée (ha) par année, calculées comme sommes groupées par tuile (sans rééchantillonnage, donc léger).

##### 8a · Étape 1 — construction de la pile (uniquement si `OVERWRITE = True` ou si elle n'existe pas encore)

In [ ]:
ensure_folder(ASSET_ROOT)
STATS_ASSET = f"{ASSET_ROOT}/stats_{RUN_TAG}"
EVENTS_ASSET = f"{ASSET_ROOT}/events_{RUN_TAG}"
GRID_ASSET = f"{ASSET_ROOT}/grid1km_tbl_{RUN_TAG}"

if BUILD_STACK and STACK_EXISTS:                      # reconstruction demandée : nécessite l'accès en écriture à la pile
    try:
        ee.data.deleteAsset(STACK_ASSET)
        print(f"ancienne pile {STACK_ASSET} supprimée")
    except Exception as e:
        print(f"⚠ impossible de remplacer {STACK_ASSET} ({str(e)[:80]}) — la pile existante est conservée")
        BUILD_STACK = False
if BUILD_STACK:
    stack_live = ee.Image.cat([
        drv_idx, drv_date.rename("t"), zone, PA_IMG, CONC_IMG, ADM_IMG,
        edge_km.multiply(1000).round().rename("edge_m"),
        road_km.multiply(1000).round().rename("road_m"),
        river_km.multiply(1000).round().rename("river_m")]).updateMask(dist_mask).toInt32()
    start_export("stack", lambda: ee.batch.Export.image.toAsset(
        image=stack_live, description=f"stack_{RUN_TAG}", assetId=STACK_ASSET, region=STUDY_REGION,
        scale=10, crs="EPSG:4326", maxPixels=1e13, pyramidingPolicy={".default": "sample"}),
        STACK_ASSET, overwrite=True)
else:
    t = asset_time(STACK_ASSET)
    print(f"Utilisation de la pile partagée {STACK_ASSET}" + (f" (mise à jour le {t:%Y-%m-%d %H:%M} UTC)" if t is not None else " — ⚠ introuvable"))

##### 8b · Étape 2 — statistiques, événements et grille dans votre dossier

In [ ]:
import time
def wait_for(assets, label):
    """Attend tant que les exports des `assets` sont en cours (ils continuent même si ce noyau s'arrête)."""
    t0 = time.time()
    while True:
        missing = [a for a in assets if asset_type(a) is None]
        if not missing:
            return True
        run = active_tasks()
        busy = [a for a in missing if a.split("/")[-1] in run]
        if not busy:
            print(f"\n⚠ {label} : {', '.join(a.split('/')[-1] for a in missing)} ni terminé(s) ni en cours — vérifiez la liste des tâches")
            return False
        if time.time() - t0 > WAIT_MAX_H * 3600:
            print(f"\n⚠ {label} toujours en cours après {WAIT_MAX_H} h — exécutez à nouveau cette cellule plus tard"); return False
        print(f"{pd.Timestamp.now():%H:%M}  en attente de {label} : {', '.join(a.split('/')[-1] for a in busy)} "
              f"— prochaine vérification dans {WAIT_POLL_MIN} min", end="\r")
        time.sleep(WAIT_POLL_MIN * 60)

if not wait_for([STACK_ASSET], "la pile"):
    raise RuntimeError("La pile n'est pas disponible — l'étape 2 ne peut pas commencer")

NBE, NBA, NZ = len(EDGE_BINS_KM) + 1, len(ACCESS_BINS_KM) + 1, len(ZONE_CODES)
FIELDS = [("year", N_YEARS), ("month", 12), ("driver", ND), ("zone", NZ),
          ("edge_bin", NBE), ("road_bin", NBA), ("river_bin", NBA)]
NPA, NCO, NAD = len(PA_NAMES) + 1, len(CONC_NAMES) + 1, len(ADM_NAMES) + 1
PROT_FIELDS = [("year", N_YEARS), ("driver", ND), ("zone", NZ), ("pa", NPA), ("conc", NCO), ("adm", NAD)]
ZP_FIELDS = [("zone", NZ), ("pa", NPA), ("conc", NCO), ("adm", NAD)]
for nm, f in [("principal", FIELDS), ("protection/administratif", PROT_FIELDS)]:
    cap = int(np.prod([s for _, s in f]))
    assert cap < 2**31, f"espace de codes {nm} {cap:,} trop grand — trop de polygones ; regroupez-les par un champ de nom plus général"

if True:
    st = ee.Image(STACK_ASSET)
    t = st.select("t"); yr = t.divide(1000).floor().add(2000).int()
    bands = {"year": yr.subtract(START_YEAR), "month": doy_to_month(yr, t.mod(1000).int()).subtract(1),
             "driver": st.select("driver"), "zone": st.select("zone").subtract(1),
             "edge_bin": bin_index(st.select("edge_m").divide(1000), EDGE_BINS_KM),
             "road_bin": bin_index(st.select("road_m").divide(1000), ACCESS_BINS_KM),
             "river_bin": bin_index(st.select("river_m").divide(1000), ACCESS_BINS_KM),
             "pa": st.select("pa"), "conc": st.select("conc"), "adm": st.select("adm")}
    dmask = st.select("driver").mask()
    area = ee.Image.pixelArea().divide(1e4).rename("area")
    dist_stack = area.addBands(encode(bands, FIELDS).rename("code")).updateMask(dmask)
    prot_stack = area.addBands(encode(bands, PROT_FIELDS).rename("code")).updateMask(dmask)
    zone_stack = area.addBands(encode({"zone": zone.subtract(1), "pa": PA_IMG, "conc": CONC_IMG, "adm": ADM_IMG},
                                      ZP_FIELDS).rename("code")).updateMask(study)

    # stats : superficie perturbée par code (10 m) + superficies des zones (résolution de l'occupation du sol)
    lc_scale = LC_PROJ.nominalScale().getInfo()
    def reduce_tile(tile):
        g, tid = tile.geometry(), tile.get("tid")
        grp = ee.Reducer.sum().group(1, "code")
        to_fc = lambda groups, kind: ee.FeatureCollection(ee.List(groups).map(
            lambda d: ee.Feature(g.centroid(100), ee.Dictionary(d)).set({"kind": kind, "tid": tid})))
        rd = dist_stack.reduceRegion(grp, g, scale=10, maxPixels=1e13, tileScale=TILE_SCALE)
        rp = prot_stack.reduceRegion(grp, g, scale=10, maxPixels=1e13, tileScale=TILE_SCALE)
        rz = zone_stack.reduceRegion(grp, g, scale=lc_scale, maxPixels=1e13, tileScale=TILE_SCALE)
        return (to_fc(rd.get("groups"), "dist").merge(to_fc(rp.get("groups"), "prot"))
                .merge(to_fc(rz.get("groups"), "zone")))
    start_export("stats", lambda: ee.batch.Export.table.toAsset(
        collection=tiles_fc.map(reduce_tile).flatten(), description=f"stats_{RUN_TAG}", assetId=STATS_ASSET), STATS_ASSET, newer_than=STACK_ASSET)

    # événements
    EV_CATS = ["driver", "zone", "edge_bin", "road_bin", "river_bin", "pa", "conc", "adm"]
    ev_img = (ee.Image(1).int().rename("seg").addBands(t.rename("t_first"))
              .addBands([ee.Image(bands[k]).int().rename(k) for k in EV_CATS]).updateMask(dmask))
    ev_red = ee.Reducer.min().unweighted().setOutputs(["t_first"])
    for k in EV_CATS:
        ev_red = ev_red.combine(ee.Reducer.mode().unweighted().setOutputs([k]), "", False)
    def tile_events(tile):
        v = ev_img.reduceToVectors(reducer=ev_red, geometry=tile.geometry(), scale=10, geometryType="polygon",
                                   eightConnected=True, labelProperty="seg", maxPixels=1e11, tileScale=TILE_SCALE)
        return v.map(lambda f: ee.Feature(f.geometry().centroid(1), f.toDictionary(["t_first"] + EV_CATS))
                     .set({"area_ha": f.geometry().area(1).divide(1e4)}))
    start_export("events", lambda: ee.batch.Export.table.toAsset(
        collection=tiles_fc.map(tile_events).flatten(), description=f"events_{RUN_TAG}", assetId=EVENTS_ASSET), EVENTS_ASSET, newer_than=STACK_ASSET)

    # grille de 1 km sous forme de tableau : sommes groupées par (cellule, année) dans chaque tuile — sans rééchantillonnage, peu de mémoire
    NCOLS, NROWS = int(math.ceil((xmax - xmin) / GRID_DEG)), int(math.ceil((ymax - ymin) / GRID_DEG))
    assert NCOLS * NROWS * N_YEARS < 2**31, "grille trop grande — augmentez GRID_DEG"
    ll = ee.Image.pixelLonLat()
    cell = (ll.select("latitude").multiply(-1).add(ymax).divide(GRID_DEG).floor().multiply(NCOLS)
            .add(ll.select("longitude").subtract(xmin).divide(GRID_DEG).floor())).int()
    g_dist = area.addBands(cell.multiply(N_YEARS).add(yr.subtract(START_YEAR)).rename("code")).updateMask(dmask)
    g_study = area.addBands(cell.rename("code")).updateMask(study)
    def grid_tile(tile):
        g, tid = tile.geometry(), tile.get("tid")
        grp = ee.Reducer.sum().group(1, "code")
        to_fc = lambda groups, kind: ee.FeatureCollection(ee.List(groups).map(
            lambda d: ee.Feature(g.centroid(100), ee.Dictionary(d)).set({"kind": kind})))
        rd = g_dist.reduceRegion(grp, g, scale=10, maxPixels=1e13, tileScale=TILE_SCALE)
        rs = g_study.reduceRegion(grp, g, scale=lc_scale, maxPixels=1e13, tileScale=TILE_SCALE)
        return to_fc(rd.get("groups"), "dist").merge(to_fc(rs.get("groups"), "study"))
    start_export("grid", lambda: ee.batch.Export.table.toAsset(
        collection=tiles_fc.map(grid_tile).flatten(), description=f"grid1km_tbl_{RUN_TAG}", assetId=GRID_ASSET), GRID_ASSET, newer_than=STACK_ASSET)

    if wait_for([STATS_ASSET, EVENTS_ASSET, GRID_ASSET], "les sorties"):
        print("\n✓ statistiques, événements et grille prêts — passez à la partie B")

#### 9 · État
Les tâches de cette exécution encore actives (y compris celles lancées avant un redémarrage du noyau), et l'état de la pile et de vos sorties.

In [ ]:
run = active_tasks()   # inclut les tâches lancées avant un redémarrage du noyau
mine = {d: v for d, v in run.items() if d and RUN_TAG in d}
if mine:
    display(pd.DataFrame([{"tâche": d, "état": v[0], "id": v[1]} for d, v in mine.items()]).set_index("tâche"))
if TASKS:
    display(pd.DataFrame([{"export": n, "état": t.status().get("state"), "erreur": t.status().get("error_message", "")}
                          for n, t in TASKS.items()]).set_index("export"))
display(pd.DataFrame({"asset": [STACK_ASSET, STATS_ASSET, EVENTS_ASSET, GRID_ASSET],
                      "prêt": ["✓" if asset_type(a) else "…" for a in (STACK_ASSET, STATS_ASSET, EVENTS_ASSET, GRID_ASSET)],
                      "mis à jour (UTC)": [f"{t:%Y-%m-%d %H:%M}" if t is not None else "–"
                                        for t in map(asset_time, (STACK_ASSET, STATS_ASSET, EVENTS_ASSET, GRID_ASSET))]},
                     index=["pile partagée", "statistiques", "événements", "grille"]))

### Partie B · Analyse
#### 10 · Chargement des résultats
Chaque tableau est téléchargé une seule fois en CSV (uniquement les colonnes nécessaires) et mis en cache dans `outputs_peat_disturbance/ee_tables/` ; les exécutions suivantes lisent la copie locale. Mettez `REFRESH_TABLES = True` au §1 pour les télécharger à nouveau après un réexport.

In [ ]:
raw = table_df(STATS_ASSET, ["kind", "code", "sum"])
d = raw[raw.kind == "dist"]
D = decode(d["code"], FIELDS); D["area_ha"] = d["sum"].to_numpy()
D = D.groupby([n for n, _ in FIELDS], as_index=False).area_ha.sum()
D["year"] = D.year + START_YEAR; D["month"] = D.month + 1
D["driver"] = D.driver.map(dict(enumerate(DRIVER_LABELS)))
D["zone"] = D.zone.map(dict(enumerate(ZONE_ORDER)))
D["access_bin"] = D[["road_bin", "river_bin"]].min(axis=1)       # le plus proche entre route et rivière
NAMES = {"pa": ["— hors aire protégée"] + PA_NAMES, "conc": ["— hors concession"] + CONC_NAMES,
         "adm": ["— aucune unité administrative"] + ADM_NAMES}
def unpack(kind, fields):
    x = raw[raw.kind == kind]
    df = decode(x["code"], fields); df["area_ha"] = x["sum"].to_numpy()
    df = df.groupby([n for n, _ in fields], as_index=False).area_ha.sum()
    if "year" in df: df["year"] = df.year + START_YEAR
    if "driver" in df: df["driver"] = df.driver.map(dict(enumerate(DRIVER_LABELS)))
    df["zone"] = df.zone.map(dict(enumerate(ZONE_ORDER)))
    for k in ("pa", "conc", "adm"):
        if k in df: df[k] = df[k].map(dict(enumerate(NAMES[k])))
    return df
P = unpack("prot", PROT_FIELDS)          # superficie perturbée par protection / unité administrative
ZP = unpack("zone", ZP_FIELDS)           # zone d'étude par protection / unité administrative
Z = ZP.groupby("zone").area_ha.sum().reindex(ZONE_ORDER)

EV = table_df(EVENTS_ASSET, ["t_first", "driver", "zone", "edge_bin", "road_bin", "river_bin",
                              "pa", "conc", "adm", "area_ha", ".geo"])
EV_COLS = ["driver", "zone", "edge_bin", "road_bin", "river_bin", "pa", "conc", "adm"]
miss = EV[EV_COLS].isna().sum()
if miss.any():
    print("Événements avec une catégorie manquante (conservés, affichés comme manquants dans les tableaux) :"); display(miss[miss > 0].to_frame("événements"))
for c in EV_COLS:
    EV[c] = EV[c].round().astype("Int64")
for k in ("pa", "conc", "adm"):
    EV[k] = EV[k].map(dict(enumerate(NAMES[k])))
EV["driver"] = EV.driver.map(dict(enumerate(DRIVER_LABELS)))
EV["zone"] = (EV.zone).map(dict(enumerate(ZONE_ORDER)))
EV["access_bin"] = EV[["road_bin", "river_bin"]].min(axis=1)
EV = EV[EV.t_first.notna()].copy()
EV["date"] = EV.t_first.apply(yydoy_to_ts)
EV["year"], EV["month"] = EV.date.dt.year, EV.date.dt.month
SIZE_LABELS = [f"petit (<{SIZE_CLASSES_HA[0]:g} ha)", f"moyen ({SIZE_CLASSES_HA[0]:g}–{SIZE_CLASSES_HA[1]:g} ha)",
               f"grand (≥{SIZE_CLASSES_HA[1]:g} ha)"]
EV["size_class"] = pd.cut(EV.area_ha, [0] + SIZE_CLASSES_HA + [np.inf], right=False, labels=SIZE_LABELS)
EV["below_mmu"] = EV.area_ha < MMU_HA

for df in (D, EV, P):
    df["with_fire"] = df.driver.isin(PARENT)
    df["driver_group"] = df.driver.map(lambda d: PARENT.get(d, d))
GROUPS = [g for g in DRIVER_GROUP_COLORS if g in set(D.driver_group)]
GCOL = [DRIVER_GROUP_COLORS[g] for g in GROUPS]
ZCOL = [ZONE_COLORS[z] for z in ZONE_ORDER]
PROVISIONAL_FROM = (pd.Timestamp(LATEST_ALERT_DATE) - pd.DateOffset(months=3)).to_period("M")
FULL_YEARS = [y for y in YEARS if pd.Period(f"{y}-12", "M") < PROVISIONAL_FROM]

D.to_csv(LOCAL_OUT / f"disturbance_stats_{RUN_TAG}.csv", index=False)
P.to_csv(LOCAL_OUT / f"disturbance_by_pa_concession_admin_{RUN_TAG}.csv", index=False)
EV.to_csv(LOCAL_OUT / f"events_{RUN_TAG}.csv", index=False)
print(f"{D.area_ha.sum():,.0f} ha perturbés sur {Z.sum():,.0f} ha de zone d'étude · {len(EV):,} événements · "
      f"dernière alerte {LATEST_ALERT_DATE} · années complètes {FULL_YEARS}")

#### 11 · Perturbations et facteurs par classe d'occupation du sol

In [ ]:
area_tbl = pd.DataFrame({"superficie de la classe (ha)": Z, "perturbé (ha)": D.groupby("zone").area_ha.sum().reindex(ZONE_ORDER)})
area_tbl["% de la classe perturbé"] = area_tbl["perturbé (ha)"] / area_tbl["superficie de la classe (ha)"] * 100
area_tbl["part des perturbations (%)"] = pct(area_tbl["perturbé (ha)"])
area_tbl.loc["total"] = [area_tbl.iloc[:, 0].sum(), area_tbl.iloc[:, 1].sum(),
                         area_tbl.iloc[:, 1].sum() / area_tbl.iloc[:, 0].sum() * 100, 100]
print("Zone d'étude et perturbations par classe"); display(area_tbl.round(2))

dz = D.pivot_table(index="driver", columns="zone", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=ZONE_ORDER, fill_value=0)
dz["total"] = dz.sum(axis=1)
dz = dz.sort_values("total", ascending=False)
print("Superficie perturbée (ha) par facteur et par classe"); display(dz.round(1))
print("Part des perturbations de chaque classe (%)"); display((dz[ZONE_ORDER] / dz[ZONE_ORDER].sum() * 100).round(1))

dg = D.pivot_table(index="zone", columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(index=ZONE_ORDER, columns=GROUPS, fill_value=0)
dy = D.pivot_table(index=["zone", "year"], columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=GROUPS, fill_value=0)
n_pan = 2 + len(ZONE_ORDER)
fig, axes = plt.subplots(math.ceil(n_pan / 2), 2, figsize=(16, 5.5 * math.ceil(n_pan / 2)), squeeze=False)
axes = axes.flatten()
dg.plot.bar(stacked=True, ax=axes[0], color=GCOL, legend=False, rot=15)
axes[0].set_ylabel("Superficie perturbée (ha)"); axes[0].set_title("Superficie par classe et par facteur")
dg.div(dg.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], color=GCOL, legend=False, rot=15)
axes[1].set_ylabel("% des perturbations de la classe"); axes[1].set_title("Parts des facteurs")
for ax, z in zip(axes[2:], ZONE_ORDER):
    if z in dy.index.get_level_values(0):
        dy.loc[z].plot.bar(stacked=True, ax=ax, color=GCOL, legend=False, rot=0)
    ax.set_title(f"{z} — par année"); ax.set_ylabel("Superficie perturbée (ha)"); ax.set_xlabel("")
for ax in axes[:2]:
    ax.set_xlabel("")
handles = [Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)]
if n_pan % 2:                                   # panneau libre → légende
    axes[-1].axis("off"); axes[-1].legend(handles=handles, loc="center", fontsize=11, title="Facteur", frameon=False)
else:
    fig.legend(handles=handles, loc="lower center", ncol=min(len(GROUPS), 5), bbox_to_anchor=(0.5, -0.04), fontsize=10)
plt.tight_layout(); save(fig, "drivers_by_class"); plt.show()

fire = D[D.driver_group.isin(PARENT.values())]
if len(fire):
    print("Défrichement agricole signalé « avec feu » (% de la superficie de ce groupe, par classe)")
    display((fire[fire.with_fire].groupby(["driver_group", "zone"]).area_ha.sum()
             / fire.groupby(["driver_group", "zone"]).area_ha.sum() * 100).unstack().reindex(columns=ZONE_ORDER).round(1))

#### 12 · Aires protégées, concessions forestières et unités administratives
Base surfacique. La « zone d'étude » est la superficie des trois classes d'occupation du sol à l'intérieur de chaque polygone. Les concessions sont présentées par attributaire (`attributai`).

In [ ]:
def status(df):
    return np.select([(df.pa != NAMES["pa"][0]) & (df.conc != NAMES["conc"][0]), df.pa != NAMES["pa"][0],
                      df.conc != NAMES["conc"][0]], ["AP + concession", "aire protégée", "concession"], "aucune")
P["status"], ZP["status"], EV["status"] = status(P), status(ZP), status(EV)
STATUS_COLORS = {"aucune": "#d9d9d9", "aire protégée": "#6a51a3", "concession": "#bf812d", "AP + concession": "#e7298a"}
SORDER = [s for s in STATUS_COLORS if s in set(ZP.status)]

st_tbl = pd.DataFrame({"zone d'étude (ha)": ZP.groupby("status").area_ha.sum(),
                       "perturbé (ha)": P.groupby("status").area_ha.sum()}).reindex(SORDER).fillna(0)
st_tbl["% de la zone d'étude perturbé"] = st_tbl["perturbé (ha)"] / st_tbl["zone d'étude (ha)"] * 100
st_tbl["part des perturbations (%)"] = pct(st_tbl["perturbé (ha)"])
st_tbl["part des événements (%)"] = pct(EV.status.value_counts()).reindex(SORDER)
print("Perturbations par statut de protection"); display(st_tbl.round(2))
print("% de chaque classe perturbé, par statut de protection")
display((P.pivot_table(index="status", columns="zone", values="area_ha", aggfunc="sum")
         / ZP.pivot_table(index="status", columns="zone", values="area_ha", aggfunc="sum") * 100)
        .reindex(index=SORDER, columns=ZONE_ORDER).round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sz = P.pivot_table(index="zone", columns="status", values="area_ha", aggfunc="sum", fill_value=0).reindex(index=ZONE_ORDER, columns=SORDER, fill_value=0)
sz.div(sz.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, ax=axes[0], color=[STATUS_COLORS[c] for c in SORDER])
axes[0].set_xlabel("% de la superficie perturbée"); axes[0].set_title("Statut de protection des perturbations, par classe")
sd_ = P.pivot_table(index="status", columns="driver_group", values="area_ha", aggfunc="sum", fill_value=0).reindex(index=SORDER, columns=GROUPS, fill_value=0)
sd_.div(sd_.sum(axis=1), axis=0).mul(100).plot.barh(stacked=True, ax=axes[1], color=GCOL)
axes[1].set_xlabel("% de la superficie perturbée"); axes[1].set_title("Facteurs à l'intérieur / à l'extérieur des AP et des concessions")
axes[1].legend(bbox_to_anchor=(1, 1), fontsize=7)
plt.tight_layout(); save(fig, "protection_status"); plt.show()

def per_polygon(key, what, fname, n=15):
    dist = P[P[key] != NAMES[key][0]]
    if dist.empty and ZP[ZP[key] != NAMES[key][0]].empty:
        print(f"Aucun(e) {what} dans la zone d'étude"); return None
    t = pd.DataFrame({"zone d'étude (ha)": ZP[ZP[key] != NAMES[key][0]].groupby(key).area_ha.sum(),
                      "perturbé (ha)": dist.groupby(key).area_ha.sum()}).fillna(0)
    t["% perturbé"] = t["perturbé (ha)"] / t["zone d'étude (ha)"].replace(0, np.nan) * 100
    top = dist.groupby([key, "driver_group"]).area_ha.sum().reset_index().sort_values("area_ha").drop_duplicates(key, keep="last")
    t["facteur principal"] = top.set_index(key).driver_group.reindex(t.index).fillna("–")
    t = t.sort_values("perturbé (ha)", ascending=False)
    t.to_csv(LOCAL_OUT / f"{fname}_{RUN_TAG}.csv")
    print(f"Perturbations par {what}"); display(t.round(2))
    sub = t.head(n).index
    bars = (dist[dist[key].isin(sub)].pivot_table(index=key, columns="driver_group", values="area_ha", aggfunc="sum",
            fill_value=0).reindex(index=sub[::-1], columns=GROUPS, fill_value=0))
    fig, axes = plt.subplots(1, 2, figsize=(15, max(3, 0.35 * len(sub) + 1)), sharey=True)
    bars.plot.barh(stacked=True, ax=axes[0], color=GCOL, legend=False)
    axes[0].set_xlabel("Superficie perturbée (ha)"); axes[0].set_title(f"{what.capitalize()} : les {len(sub)} plus perturbé(e)s"); axes[0].set_ylabel("")
    t.loc[sub[::-1], "% perturbé"].plot.barh(ax=axes[1], color="#d7191c")
    axes[1].set_xlabel("% de la zone d'étude perturbé"); axes[1].set_title("Par rapport à la zone d'étude")
    fig.legend(handles=[Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)], loc="lower center",
               ncol=min(len(GROUPS), 5), bbox_to_anchor=(0.5, -0.12), fontsize=7)
    plt.tight_layout(); save(fig, fname); plt.show()
    return t

PA_T = per_polygon("pa", "aire protégée", "by_protected_area")
CONC_T = per_polygon("conc", "concession forestière (attributaire)", "by_concession")
ADM_T = per_polygon("adm", "unité administrative", "by_admin_unit", n=20)

#### 13 · Taille des événements
Les événements sont des pixels perturbés connexes (8-connexité) ; classes de taille comme dans Nesha et al. (petit < 0,5 ha, moyen 0,5–2 ha, grand ≥ 2 ha).

In [ ]:
summ = EV.groupby("zone").area_ha.describe(percentiles=[.5, .9]).reindex(ZONE_ORDER)
summ.loc["tout"] = EV.area_ha.describe(percentiles=[.5, .9])
print(f"{len(EV):,} événements · {EV.below_mmu.mean() * 100:.1f} % sous {MMU_HA} ha (fragments)")
display(summ.rename(columns={"count": "événements", "mean": "moyenne ha", "50%": "médiane ha", "90%": "p90 ha", "max": "max ha"})
        [["événements", "moyenne ha", "médiane ha", "p90 ha", "max ha"]].round(2))

sc = EV.groupby("size_class", observed=False).agg(events=("area_ha", "size"), area_ha=("area_ha", "sum"))
sc["% des événements"] = pct(sc.events); sc["% de la superficie"] = pct(sc.area_ha)
display(sc.rename(columns={"events": "événements"}).round(1))
print("Classes de taille des événements par facteur (% de la superficie de chaque facteur)")
display(pd.crosstab(EV.driver_group, EV.size_class, values=EV.area_ha, aggfunc="sum", normalize="index")
        .mul(100).reindex(GROUPS).round(1))

fig, axes = plt.subplots(3, 1, figsize=(12, 16))
bins = np.logspace(np.log10(max(EV.area_ha.min(), 0.01)), np.log10(EV.area_ha.max() + 0.01), 40)
for z, c in zip(ZONE_ORDER, ZCOL):
    axes[0].hist(EV.loc[EV.zone == z, "area_ha"], bins=bins, histtype="step", lw=1.5, color=c, label=z)
axes[0].set_xscale("log"); plain_log(axes[0].xaxis); axes[0].set_xlabel("Superficie de l'événement (ha, échelle log)"); axes[0].set_ylabel("Événements"); axes[0].legend(fontsize=8)
axes[0].set_title("Distribution de la taille des événements")
for t in SIZE_CLASSES_HA: axes[0].axvline(t, color="grey", ls=":")
sd = EV.pivot_table(index="size_class", columns="driver_group", values="area_ha", aggfunc="sum", observed=False,
                    fill_value=0).reindex(columns=GROUPS, fill_value=0)
sd.div(sd.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], color=GCOL, rot=0)
axes[1].set_ylabel("% de la superficie"); axes[1].set_xlabel(""); axes[1].set_title("Parts des facteurs par taille d'événement"); axes[1].legend(fontsize=7)
EV.boxplot(column="area_ha", by="driver_group", ax=axes[2], rot=30, showfliers=False)
axes[2].set_yscale("log"); plain_log(axes[2].yaxis); axes[2].set_ylabel("Superficie de l'événement (ha, échelle log)"); axes[2].set_title("Taille des événements par facteur"); axes[2].set_xlabel("")
plt.suptitle(""); plt.tight_layout(); save(fig, "event_sizes"); plt.show()

#### 14 · Profils temporels par mois
Part des perturbations de chaque année par mois ; les mois au-dessus de `TEMPORAL_HOTSPOT_PCT` sont des points chauds temporels. Les lignes pointillées correspondent aux années partielles ou provisoires.

In [ ]:
mt = D.pivot_table(index="year", columns="month", values="area_ha", aggfunc="sum", fill_value=0).reindex(columns=range(1, 13), fill_value=0)
mp = mt.div(mt.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for y in mt.index:
    ls = "-" if y in FULL_YEARS else "--"
    axes[0].plot(range(1, 13), mt.loc[y], marker="o", ls=ls, label=str(y))
    axes[1].plot(range(1, 13), mp.loc[y], marker="o", ls=ls, label=str(y))
axes[1].axhline(TEMPORAL_HOTSPOT_PCT, color="grey", ls=":")
axes[0].set_ylabel("Superficie perturbée (ha)"); axes[1].set_ylabel("% des perturbations de l'année")
for a in axes: a.set_xticks(range(1, 13)); a.set_xlabel("Mois")
axes[0].legend(ncol=3, fontsize=8); plt.tight_layout(); save(fig, "monthly"); plt.show()

fig, ax = plt.subplots(figsize=(10, 0.6 * len(mp) + 1))
im = ax.imshow(mp.to_numpy(), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(12)); ax.set_xticklabels(range(1, 13)); ax.set_yticks(range(len(mp))); ax.set_yticklabels(mp.index)
for (i, j), v in np.ndenumerate(mp.to_numpy()):
    if v > TEMPORAL_HOTSPOT_PCT: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="% de l'année"); ax.set_title(f"Points chauds temporels (mois > {TEMPORAL_HOTSPOT_PCT} % indiqués)")
plt.tight_layout(); save(fig, "monthly_heatmap"); plt.show()

display(pd.DataFrame({"part janv.–juin (%)": mp[list(range(1, 7))].sum(axis=1).round(1),
                      "mois points chauds": [[m for m in range(1, 13) if mp.loc[y, m] > TEMPORAL_HOTSPOT_PCT] for y in mp.index]},
                     index=mp.index))

# profil mensuel par facteur (toutes années regroupées) et par classe
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pg = D[D.year.isin(FULL_YEARS)].pivot_table(index="month", columns="driver_group", values="area_ha", aggfunc="sum",
                                             fill_value=0).reindex(index=range(1, 13), columns=GROUPS, fill_value=0)
(pg / pg.sum() * 100).plot(ax=axes[0], color=GCOL, marker=".")
axes[0].set_ylabel("% de la superficie annuelle du facteur"); axes[0].set_title("Profil mensuel par facteur (années complètes)")
pz = D[D.year.isin(FULL_YEARS)].pivot_table(index="month", columns="zone", values="area_ha", aggfunc="sum",
                                             fill_value=0).reindex(index=range(1, 13), columns=ZONE_ORDER, fill_value=0)
(pz / pz.sum() * 100).plot(ax=axes[1], color=ZCOL, marker=".")
axes[1].set_ylabel("% de la superficie annuelle de la classe"); axes[1].set_title("Profil mensuel par classe (années complètes)")
for a in axes: a.set_xticks(range(1, 13)); a.set_xlabel("Mois"); a.legend(fontsize=7)
plt.tight_layout(); save(fig, "monthly_by_driver_class"); plt.show()

#### 15 · Points chauds spatiaux sur la grille de 1 km
Intensité = superficie perturbée / zone d'étude dans chaque cellule d'environ 1 km (`GRID_DEG`, toute la période). Faible < 2 %, moyenne 2–20 %, élevée > 20 % (point chaud spatial), comme dans Nesha et al. Suivent les cartes du nombre d'événements et de la taille médiane des événements par cellule, ainsi que du facteur dominant.

In [ ]:
lo, hi = INTENSITY_THRESHOLDS_PCT
NCOLS, NROWS = int(math.ceil((xmax - xmin) / GRID_DEG)), int(math.ceil((ymax - ymin) / GRID_DEG))
GEXT = (xmin, xmin + NCOLS * GRID_DEG, ymax - NROWS * GRID_DEG, ymax)
GT = table_df(GRID_ASSET, ["kind", "code", "sum"])
gd = GT[GT.kind == "dist"]
dist_cy = (pd.DataFrame({"cell": (gd.code.round().astype("int64") // N_YEARS),
                         "year": (gd.code.round().astype("int64") % N_YEARS) + START_YEAR, "ha": gd["sum"]})
           .groupby(["cell", "year"]).ha.sum().unstack(fill_value=0).reindex(columns=YEARS, fill_value=0))
study_c = GT[GT.kind == "study"].assign(cell=lambda x: x.code.round().astype("int64")).groupby("cell")["sum"].sum()
CELLS = pd.DataFrame({"study_ha": study_c}).join(dist_cy, how="outer").fillna(0)
CELLS["dist_ha_all"] = CELLS[YEARS].sum(axis=1)
CELLS["intensity_pct"] = (CELLS.dist_ha_all / CELLS.study_ha.where(CELLS.study_ha > 0) * 100).clip(upper=100)
CELLS["class"] = pd.cut(CELLS.intensity_pct, [0, lo, hi, np.inf], right=False,
                        labels=[f"faible (<{lo} %)", f"moyenne ({lo}–{hi} %)", f"élevée / point chaud (>{hi} %)"])
CELLS.loc[CELLS.dist_ha_all <= 0, "class"] = np.nan
CELLS["lon"] = xmin + (CELLS.index % NCOLS + 0.5) * GRID_DEG
CELLS["lat"] = ymax - (CELLS.index // NCOLS + 0.5) * GRID_DEG
CELLS.to_csv(LOCAL_OUT / f"grid1km_{RUN_TAG}.csv")
CLS_NAMES = list(CELLS["class"].cat.categories); CLS_COLORS = ["#fdae61", "#d7191c", "#2c007a"]

INT = CELLS.dropna(subset=["class"]).groupby("class", observed=False).agg(**{"cellules de 1 km": ("dist_ha_all", "size"),
                                                                             "ha perturbés": ("dist_ha_all", "sum")})
INT["% des cellules"] = pct(INT["cellules de 1 km"]); INT["% de la superficie perturbée"] = pct(INT["ha perturbés"])
print(f"{(CELLS.dist_ha_all > 0).sum():,} cellules perturbées sur {(CELLS.study_ha > 0).sum():,} cellules avec zone d'étude")
display(INT.round(1))
print("Les 15 cellules les plus intenses (points chauds)")
display(CELLS.sort_values("intensity_pct", ascending=False).head(15)[["lon", "lat", "study_ha", "dist_ha_all", "intensity_pct"] + YEARS].round(2))

def to_arr(series):
    a = np.full(NROWS * NCOLS, np.nan)
    v = series.dropna(); v = v[(v.index >= 0) & (v.index < NROWS * NCOLS)]
    a[v.index.to_numpy()] = v.to_numpy()
    return a.reshape(NROWS, NCOLS)
study_arr = to_arr(CELLS.study_ha.where(CELLS.study_ha > 0))
cls_arr = to_arr(CELLS["class"].cat.codes.where(CELLS["class"].notna()).astype(float) + 1)
EXTENT = GEXT

borders = [r for f in gaul0.map(lambda f: f.simplify(2000)).getInfo()["features"] for r in geom_rings(f["geometry"])]
def outlines(asset):
    if asset_type(asset) != "TABLE":
        return []
    gj = ee.FeatureCollection(asset).filterBounds(EXTENT_BOX).map(lambda f: ee.Feature(f.geometry().simplify(500))).getInfo()
    return [r for f in gj["features"] for r in geom_rings(f["geometry"])]
PA_LINES, CONC_LINES = outlines(PA_ASSET), outlines(CONCESSIONS_ASSET)
def basemap(ax, title, context=True):
    for b in borders: ax.plot(b[:, 0], b[:, 1], color="black", lw=0.6)
    if context:
        for l in PA_LINES: ax.plot(l[:, 0], l[:, 1], color="#6a51a3", lw=0.7)
        for l in CONC_LINES: ax.plot(l[:, 0], l[:, 1], color="#8c510a", lw=0.4, ls="--")
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=7); ax.set_facecolor("#f7f7f7")

fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#d9f0d3"]), interpolation="nearest")
ax.imshow(cls_arr, extent=EXTENT, cmap=ListedColormap(CLS_COLORS), vmin=0.5, vmax=3.5, interpolation="nearest")
basemap(ax, f"Intensité des perturbations par cellule d'environ 1 km, {START_YEAR}–{END_YEAR}")
ax.legend(handles=[Patch(color="#d9f0d3", label="zone d'étude, sans perturbation")] +
          [Patch(color=c, label=n) for n, c in zip(CLS_NAMES, CLS_COLORS)] +
          [Line2D([], [], color="#6a51a3", lw=0.9, label="aire protégée"),
           Line2D([], [], color="#8c510a", lw=0.7, ls="--", label="concession forestière")], loc="lower left", fontsize=8)
plt.tight_layout(); save(fig, "map_hotspots"); plt.show()

ncol = min(3, N_YEARS); nrow = math.ceil(N_YEARS / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4.5 * nrow), squeeze=False)
for ax, y in zip(axes.flat, YEARS):
    sh = to_arr((CELLS[y] / CELLS.study_ha.where(CELLS.study_ha > 0) * 100).where(CELLS[y] > 0))
    ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
    im = ax.imshow(sh, extent=EXTENT, cmap="inferno_r", vmin=0, vmax=hi / 2, interpolation="nearest")
    basemap(ax, f"{y}" + ("" if y in FULL_YEARS else " (partielle)"), context=False)
for ax in axes.flat[N_YEARS:]: ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.5, label="% de la zone d'étude perturbé dans l'année")
save(fig, "map_hotspots_by_year"); plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(CELLS.intensity_pct[CELLS.dist_ha_all > 0], bins=np.arange(0, 101, 1), color="teal")
for t in INTENSITY_THRESHOLDS_PCT: ax.axvline(t, color="red", ls="--", lw=0.8)
ax.set_yscale("log"); plain_log(ax.yaxis); ax.set_xlabel("Intensité (% de la zone d'étude perturbé par cellule d'environ 1 km)"); ax.set_ylabel("Cellules (échelle log)")
plt.tight_layout(); save(fig, "hist_intensity"); plt.show()

# événements sur la grille d'environ 1 km : nombre d'événements et taille médiane des événements par cellule
from matplotlib.colors import LogNorm
col_i = np.floor((EV.lon - xmin) / GRID_DEG); row_i = np.floor((ymax - EV.lat) / GRID_DEG)
inside = col_i.between(0, NCOLS - 1) & row_i.between(0, NROWS - 1)          # événements dans l'emprise de la grille
evg = EV[inside].assign(cell=(row_i[inside] * NCOLS + col_i[inside]).astype("int64")).groupby("cell").agg(events=("area_ha", "size"), median_ha=("area_ha", "median"),
                                                 total_ha=("area_ha", "sum"))
print(f"Événements par cellule d'environ 1 km : médiane {evg.events.median():.0f}, max {evg.events.max():,} · "
      f"{(evg.events >= 10).sum():,} cellules avec ≥ 10 événements")
for col, title, cmap, norm, lab in [
        ("events", "Nombre d'événements de perturbation par cellule d'environ 1 km", "magma_r", LogNorm(1, max(evg.events.max(), 2)), "événements (échelle log)"),
        ("median_ha", "Taille médiane des événements par cellule d'environ 1 km", "viridis_r", LogNorm(max(evg.median_ha.min(), 0.01), max(evg.median_ha.max(), 0.1)), "taille médiane des événements (ha, échelle log)")]:
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]), interpolation="nearest")
    im = ax.imshow(to_arr(evg[col]), extent=EXTENT, cmap=cmap, norm=norm, interpolation="nearest")
    basemap(ax, f"{title}, {START_YEAR}–{END_YEAR}")
    cb = fig.colorbar(im, ax=ax, shrink=0.6, label=lab); plain_log(cb.ax.yaxis)
    plt.tight_layout(); save(fig, f"map_{col}_per_cell"); plt.show()

# facteur dominant par cellule de DOM_CELL_DEG (d'après les événements)
nx, ny = int(round((xmax - xmin) / DOM_CELL_DEG)), int(round((ymax - ymin) / DOM_CELL_DEG))
cells = EV.assign(ix=((EV.lon - xmin) / DOM_CELL_DEG).astype(int).clip(0, nx - 1),
                  iy=((ymax - EV.lat) / DOM_CELL_DEG).astype(int).clip(0, ny - 1))
by = cells.groupby(["iy", "ix", "driver_group"]).area_ha.sum().reset_index()
dom = by.sort_values("area_ha").drop_duplicates(["iy", "ix"], keep="last")
dom_arr = np.full((ny, nx), np.nan)
dom_arr[dom.iy.to_numpy(), dom.ix.to_numpy()] = dom.driver_group.map({g: i for i, g in enumerate(GROUPS)}).to_numpy()
fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(np.where(study_arr > 0, 1, np.nan), extent=EXTENT, cmap=ListedColormap(["#e5f5e0"]))
ax.imshow(dom_arr, extent=EXTENT, cmap=ListedColormap(GCOL), vmin=-0.5, vmax=len(GROUPS) - 0.5, interpolation="nearest")
basemap(ax, f"Facteur dominant par cellule de {DOM_CELL_DEG}°")
ax.legend(handles=[Patch(color=c, label=g) for g, c in zip(GROUPS, GCOL)], loc="lower left", fontsize=8)
plt.tight_layout(); save(fig, "map_dominant_driver"); plt.show()

#### 16 · Perturbations selon la distance au bord de la tourbière, aux routes et aux rivières
Part cumulée des événements de perturbation à chaque distance (comme dans Nesha et al., fig. 5). La distance au bord de la tourbière n'utilise que les événements des classes de tourbe ; les distances aux routes et aux rivières utilisent tous les événements.

In [ ]:
EVp = EV[EV.zone != ZONE_BUFFER]
cur = pd.DataFrame({"bord de la tourbière (classes de tourbe)": cum_share(EVp, "edge_bin", EDGE_BINS_KM)})
acc = pd.DataFrame({"route": cum_share(EV, "road_bin", ACCESS_BINS_KM),
                    "rivière": cum_share(EV, "river_bin", ACCESS_BINS_KM),
                    "route ou rivière": cum_share(EV, "access_bin", ACCESS_BINS_KM)})
print("% cumulé des événements"); display(cur.T.round(1)); display(acc.T.round(1))
print("% cumulé de la superficie perturbée (base pixels)")
display(pd.DataFrame({"bord de la tourbière (classes de tourbe)": cum_share(D[D.zone != ZONE_BUFFER], "edge_bin", EDGE_BINS_KM, "area_ha")}).T.round(1))
display(pd.DataFrame({k: cum_share(D, c, ACCESS_BINS_KM, "area_ha") for k, c in
                      [("route", "road_bin"), ("rivière", "river_bin"), ("route ou rivière", "access_bin")]}).T.round(1))

fig, axes = plt.subplots(3, 1, figsize=(12, 16))
axes[0].plot(EDGE_BINS_KM, cur.iloc[:-1, 0], marker="o", color="#e34a33", label="Bord de la tourbière")
for col, c in zip(acc.columns, ["#7f0000", "#2b4bd8", "#1b9e77"]):
    axes[0].plot(ACCESS_BINS_KM, acc[col].iloc[:-1], marker="o", color=c, label=col.capitalize())
axes[0].set_title("Tous les événements")
for ax, (lab, col, edges, df) in zip(axes[1:], [("bord de la tourbière", "edge_bin", EDGE_BINS_KM, EVp),
                                                ("route ou rivière", "access_bin", ACCESS_BINS_KM, EV)]):
    for g, c in zip(GROUPS, GCOL):
        e = df[df.driver_group == g]
        if len(e) >= 20:
            ax.plot(edges, cum_share(e, col, edges).iloc[:-1], marker=".", color=c, label=g)
    ax.set_title(f"Par facteur — distance : {lab}")
for ax in axes:
    ax.set_xlabel("Distance (km)"); ax.set_ylabel("% cumulé des événements"); ax.set_ylim(0, 101); ax.legend(fontsize=7)
plt.tight_layout(); save(fig, "distance_curves"); plt.show()

print("Seuils clés par facteur (% des événements)")
def thresholds(g):
    gp = g[g.zone != ZONE_BUFFER]
    return {"≤1 km du bord de la tourbière": cum_share(gp, "edge_bin", EDGE_BINS_KM)["≤1 km"],
            "≤3 km du bord de la tourbière": cum_share(gp, "edge_bin", EDGE_BINS_KM)["≤3 km"],
            "≤5 km d'une route": cum_share(g, "road_bin", ACCESS_BINS_KM)["≤5 km"],
            "≤5 km d'une rivière": cum_share(g, "river_bin", ACCESS_BINS_KM)["≤5 km"],
            "≤10 km d'une route ou d'une rivière": cum_share(g, "access_bin", ACCESS_BINS_KM)["≤10 km"],
            "événements": len(g)}
display(pd.DataFrame({grp: thresholds(EV[EV.driver_group == grp]) for grp in GROUPS}).T.round(1))

#### 17 · Exports pour les SIG et les tableaux de bord
Écrits dans `outputs_peat_disturbance/gis/` et `outputs_peat_disturbance/dashboard/`.

**GeoTIFF** (`EXPORT_GEOTIFF`) : `peat_disturbance_grid_<RUN_TAG>.tif`, EPSG:4326, un pixel d'environ 1 km par cellule de la grille (`GRID_DEG`), float32, NaN = pas de données. Un fichier `…_bands.csv` indique la signification des bandes et les tables de codes.

**Tableaux pour tableaux de bord** (`EXPORT_DASHBOARD_TABLES`), tous en CSV simple :
| fichier | une ligne par | coordonnées / clés de jointure |
|---|---|---|
| `events_points` | événement de perturbation | `lon`, `lat` (centre de l'événement) |
| `grid_cells_by_year` | cellule d'environ 1 km × année | `lon`, `lat` (centre de la cellule), `cell_id` |
| `grid_cells_total` | cellule d'environ 1 km (toute la période) | `lon`, `lat`, `cell_id` |
| `summary_monthly` | année × mois × facteur × classe | — |
| `summary_by_area` | année × facteur × classe × aire protégée × concession × unité administrative | les noms correspondent aux champs de nom des couches de polygones (`PA_FIELD`, `CONC_FIELD`, `ADM_FIELD`) pour les jointures |
| `study_area_by_area` | classe × aire protégée × concession × unité administrative | mêmes clés de jointure (dénominateurs du % perturbé) |

Les noms des colonnes restent en anglais (pour ne pas casser les jointures et tableaux de bord existants) ; les valeurs (facteurs, classes, statuts) sont en français.

In [ ]:
GIS_DIR = LOCAL_OUT / "gis"; GIS_DIR.mkdir(exist_ok=True)
DASH_DIR = LOCAL_OUT / "dashboard"; DASH_DIR.mkdir(exist_ok=True)

def bin_label(edges):
    return {i: lab for i, lab in enumerate([f"0–{edges[0]:g} km"] + [f"{a:g}–{b:g} km" for a, b in zip(edges[:-1], edges[1:])]
                                           + [f">{edges[-1]:g} km"])}
EDGE_LAB, ACC_LAB = bin_label(EDGE_BINS_KM), bin_label(ACCESS_BINS_KM)

# facteur principal par cellule d'environ 1 km (plus grande superficie d'événements)
ev_in = EV[inside].assign(cell=(row_i[inside] * NCOLS + col_i[inside]).astype("int64"))
main_drv = (ev_in.groupby(["cell", "driver_group"]).area_ha.sum().reset_index()
            .sort_values("area_ha").drop_duplicates("cell", keep="last").set_index("cell").driver_group)
DRV_CODE = {g: i + 1 for i, g in enumerate(GROUPS)}

# ── GeoTIFF ─────────────────────────────────────────────────────────────────
if EXPORT_GEOTIFF:
    bands = {"study_ha": to_arr(CELLS.study_ha.where(CELLS.study_ha > 0)),
             "disturbed_ha_all": to_arr(CELLS.dist_ha_all.where(CELLS.dist_ha_all > 0)),
             "intensity_pct": to_arr(CELLS.intensity_pct.where(CELLS.dist_ha_all > 0)),
             "intensity_class": cls_arr,
             **{f"disturbed_ha_{y}": to_arr(CELLS[y].where(CELLS[y] > 0)) for y in YEARS},
             "events": to_arr(evg.events.astype(float)),
             "median_event_ha": to_arr(evg.median_ha),
             "main_driver": to_arr(main_drv.map(DRV_CODE).astype(float))}
    tif = GIS_DIR / f"peat_disturbance_grid_{RUN_TAG}.tif"
    stack_arr = np.stack([b.astype("float32") for b in bands.values()])
    try:
        import rasterio
        from rasterio.transform import from_origin
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)
        import rasterio
        from rasterio.transform import from_origin
    with rasterio.open(tif, "w", driver="GTiff", height=NROWS, width=NCOLS, count=len(bands), dtype="float32",
                       crs="EPSG:4326", transform=from_origin(xmin, ymax, GRID_DEG, GRID_DEG), nodata=np.nan,
                       compress="deflate") as dst:
        dst.write(stack_arr)
        for i, name in enumerate(bands, start=1):
            dst.set_band_description(i, name)
        dst.update_tags(run=RUN_TAG, period=f"{START_YEAR}-{END_YEAR}", latest_alert=LATEST_ALERT_DATE,
                        intensity_class="1 faible, 2 moyenne, 3 élevée / point chaud",
                        main_driver="; ".join(f"{v} {k}" for k, v in DRV_CODE.items()))
    legend = pd.DataFrame({"bande": range(1, len(bands) + 1), "name": list(bands)})
    legend["signification"] = legend.name.map({
        "study_ha": "zone d'étude dans la cellule (ha)", "disturbed_ha_all": "superficie perturbée, toute la période (ha)",
        "intensity_pct": "superficie perturbée / zone d'étude (%)", "intensity_class": f"1 faible (<{lo} %), 2 moyenne ({lo}–{hi} %), 3 élevée (>{hi} %)",
        "events": "nombre d'événements (par centre d'événement)", "median_event_ha": "taille médiane des événements (ha)",
        "main_driver": "; ".join(f"{v} = {k}" for k, v in DRV_CODE.items())}).fillna("superficie perturbée dans l'année (ha)")
    legend.to_csv(GIS_DIR / f"peat_disturbance_grid_{RUN_TAG}_bands.csv", index=False)
    print(f"GeoTIFF : {tif}  ({len(bands)} bandes, {NCOLS} × {NROWS} px)"); display(legend.set_index("bande"))

# ── tableaux pour tableaux de bord ──────────────────────────────────────────
if EXPORT_DASHBOARD_TABLES:
    ev_out = pd.DataFrame({
        "event_id": range(1, len(EV) + 1), "lon": EV.lon.round(6), "lat": EV.lat.round(6),
        "date": EV.date.dt.date, "year": EV.year, "month": EV.month,
        "driver": EV.driver, "driver_group": EV.driver_group, "with_fire": EV.with_fire,
        "landcover_class": EV.zone, "area_ha": EV.area_ha.round(4), "size_class": EV.size_class.astype(str),
        "protected_area": EV.pa, "concession": EV.conc, "admin_unit": EV.adm,
        "dist_peat_edge": EV.edge_bin.map(EDGE_LAB).where(EV.zone != ZONE_BUFFER),
        "dist_road": EV.road_bin.map(ACC_LAB), "dist_river": EV.river_bin.map(ACC_LAB),
        "dist_road_or_river": EV.access_bin.map(ACC_LAB), "below_mmu": EV.below_mmu})
    cells_y = (CELLS[YEARS + ["study_ha", "lon", "lat"]].reset_index(names="cell_id")
               .melt(id_vars=["cell_id", "lon", "lat", "study_ha"], value_vars=YEARS, var_name="year", value_name="disturbed_ha"))
    cells_y = cells_y[cells_y.disturbed_ha > 0]
    cells_y["intensity_pct"] = (cells_y.disturbed_ha / cells_y.study_ha.where(cells_y.study_ha > 0) * 100).clip(upper=100)
    cells_t = (CELLS[["lon", "lat", "study_ha", "dist_ha_all", "intensity_pct", "class"]]
               .join(evg[["events", "median_ha"]]).join(main_drv.rename("main_driver"))
               .reset_index(names="cell_id").rename(columns={"dist_ha_all": "disturbed_ha", "class": "intensity_class"}))
    cells_t = cells_t[cells_t.study_ha > 0]
    monthly = (D.groupby(["year", "month", "driver", "driver_group", "zone"]).area_ha.sum().reset_index()
               .rename(columns={"zone": "landcover_class", "area_ha": "disturbed_ha"}))
    by_area = (P.groupby(["year", "driver", "driver_group", "zone", "pa", "conc", "adm"]).area_ha.sum().reset_index()
               .rename(columns={"zone": "landcover_class", "pa": "protected_area", "conc": "concession",
                                "adm": "admin_unit", "area_ha": "disturbed_ha"}))
    study_area = (ZP.rename(columns={"zone": "landcover_class", "pa": "protected_area", "conc": "concession",
                                     "adm": "admin_unit", "area_ha": "study_ha"}))
    study_area = study_area[["landcover_class", "protected_area", "concession", "admin_unit", "study_ha"]]
    out = {"events_points": ev_out, "grid_cells_by_year": cells_y, "grid_cells_total": cells_t,
           "summary_monthly": monthly, "summary_by_area": by_area, "study_area_by_area": study_area}
    for name, df in out.items():
        df.to_csv(DASH_DIR / f"{name}_{RUN_TAG}.csv", index=False, float_format="%.4f")
    display(pd.DataFrame({"lignes": {k: len(v) for k, v in out.items()},
                          "colonnes": {k: ", ".join(v.columns) for k, v in out.items()}}))
    print(f"Tableaux pour tableaux de bord dans {DASH_DIR}/ — les noms dans protected_area / concession / admin_unit correspondent aux "
          f"champs '{PA_FIELD}', '{CONC_FIELD}' et '{ADM_FIELD}' des couches de polygones (hors polygone = '— …').")

#### 18 · Remarques
* **Définition d'une perturbation.** Un pixel est compté si les cartes du WUR contiennent une alerte datée entre `START_YEAR` et `END_YEAR` ; l'alerte la plus récente et sa classe sont conservées. Les ~3 derniers mois avant la dernière alerte sont provisoires, et l'année en cours est partielle (en pointillés dans les graphiques).
* **Pas de masque forestier.** La zone d'étude correspond aux classes d'occupation du sol elles-mêmes ; l'intensité est donc la part perturbée de la zone d'étude de chaque cellule.
* **Distances.** La distance au bord de la tourbière provient de `PEAT_DIST_ASSET`. Les distances aux routes et aux rivières sont calculées à 100 m, plafonnées à 30 km, après masquage des éléments au pays choisi : les routes ou rivières situées juste de l'autre côté de la frontière ne sont donc pas prises en compte.
* **Aires protégées, concessions, unités administratives** : rastérisées par nom de polygone (`NAME` pour les aires protégées, `attributai` pour les concessions) ; là où des polygones se superposent, le dernier rastérisé l'emporte. Le masque du pays s'applique aussi à ces couches.
* **Événements** : pixels connexes (8-connexité) sur toute la période, découpés aux limites des tuiles de calcul. Les événements plus petits que `MMU_HA` sont surtout des fragments créés par les limites des classes ; ils sont signalés dans `below_mmu`.